# Churn Prediction — Exploratory Analysis

Features are built from `retail_clean.csv` using an observation window (before 2011-06-01).  
Churn label: customers classified as **At Risk** or **Lost** in the RFM segmentation.  
For the full production run (Optuna tuning + MLflow + gate check): `python run_churn.py --n-trials 50`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.churn import (
    AUC_ROC_GATE,
    PRECISION_TOP20_GATE,
    SNAPSHOT,
    load_and_preprocess,
    precision_at_top_k,
)

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

## 1. Raw Transaction Data — Overview

In [ ]:
retail = pd.read_csv(ROOT / 'data/processed/retail_clean.csv',
                     parse_dates=['InvoiceDate'])
print(f'Transactions: {len(retail):,}')
print(f'Customers:    {retail["Customer ID"].nunique():,}')
print(f'Date range:   {retail["InvoiceDate"].min().date()} to {retail["InvoiceDate"].max().date()}')
print(f'Observation window cutoff (SNAPSHOT): {SNAPSHOT.date()}')
retail.head()

In [ ]:
obs = retail[retail['InvoiceDate'] < SNAPSHOT]
print(f'Observation window transactions: {len(obs):,}')
print(f'Observation window customers:    {obs["Customer ID"].nunique():,}')

monthly = obs.groupby(obs['InvoiceDate'].dt.to_period('M'))['Revenue'].sum()
fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(ax=ax, color='steelblue')
ax.axvline(pd.Period('2011-06', 'M'), color='red', linestyle='--', label='SNAPSHOT')
ax.set_title('Monthly Revenue — Observation Window')
ax.set_ylabel('Revenue')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Churn Label — RFM Segmentation

In [ ]:
rfm = pd.read_csv(ROOT / 'data/processed/rfm_scores.csv')
print(f'RFM customers: {len(rfm):,}')
print('\nSegment distribution:')
print(rfm['Segment'].value_counts().to_string())

churn_map = {'At Risk': 'Churned', 'Lost': 'Churned',
             'Champions': 'Retained', 'Loyal Customers': 'Retained',
             'Potential Loyalists': 'Retained'}
rfm['churn_label'] = rfm['Segment'].map(churn_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
rfm['Segment'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('RFM Segment Distribution')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

rfm['churn_label'].value_counts().plot(kind='bar', ax=axes[1],
    color=['steelblue', 'salmon'])
axes[1].set_title('Churn Label (At Risk + Lost = Churned)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

churn_rate = (rfm['churn_label'] == 'Churned').mean()
print(f'\nChurn rate: {churn_rate:.1%}')

## 3. Feature Engineering

In [ ]:
X, y, cids = load_and_preprocess(ROOT / 'data/processed/retail_clean.csv')

print(f'Dataset: {X.shape[0]:,} customers, {X.shape[1]} features')
print(f'Churn rate (At Risk + Lost): {y.mean():.1%}')
print(f'\nFeatures:')
for col in X.columns:
    print(f'  {col}')

In [ ]:
X.describe().T[['mean', 'std', 'min', '50%', 'max']].round(2)

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for ax, col in zip(axes.flat, X.columns):
    for churn_val, color, label in [(0, 'steelblue', 'Retained'), (1, 'salmon', 'Churned')]:
        data = X.loc[y == churn_val, col]
        ax.hist(data.clip(upper=data.quantile(0.99)), bins=30,
                alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col, fontsize=8)
    ax.legend(fontsize=6)
for ax in axes.flat[len(X.columns):]:
    ax.set_visible(False)
fig.suptitle('Feature Distributions by Churn Status', fontsize=13)
plt.tight_layout()
plt.show()

### Correlation with Churn Label

In [ ]:
corr = X.corrwith(y).sort_values(key=abs, ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['salmon' if v > 0 else 'steelblue' for v in corr]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Churn Label')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

## 4. Train / Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X.values, y.values, test_size=0.20, stratify=y.values, random_state=42
)
print(f'Train: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'Train churn rate: {y_tr.mean():.3f}')
print(f'Test  churn rate: {y_te.mean():.3f}')

## 5. Quick Model Training (Default Params)

Single XGBoost run with fixed params for exploration.  
Full Optuna tuning (50 trials) + MLflow + gate check: `python run_churn.py --n-trials 50`

In [ ]:
spw   = float((y_tr == 0).sum() / (y_tr == 1).sum())
model = XGBClassifier(
    max_depth=6, learning_rate=0.05, n_estimators=400,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw, tree_method='hist',
    random_state=42, n_jobs=-1, verbosity=0,
)
model.fit(X_tr, y_tr)
y_prob = model.predict_proba(X_te)[:, 1]

auc_roc = roc_auc_score(y_te, y_prob)
p_top20 = precision_at_top_k(y_te, y_prob, k=0.20)

status_auc = 'PASS' if auc_roc >= AUC_ROC_GATE        else 'FAIL'
status_p20 = 'PASS' if p_top20 >= PRECISION_TOP20_GATE else 'FAIL'

print(f'AUC-ROC:          {auc_roc:.4f}   (gate >= {AUC_ROC_GATE})  [{status_auc}]')
print(f'Precision@top20:  {p_top20:.4f}   (gate >= {PRECISION_TOP20_GATE})  [{status_p20}]')

## 6. SHAP Explainability

In [ ]:
X_te_df   = pd.DataFrame(X_te, columns=X.columns.tolist())
explainer = shap.TreeExplainer(model)
sv        = explainer.shap_values(X_te_df)

shap.summary_plot(sv, X_te_df, feature_names=X.columns.tolist())

In [ ]:
shap.summary_plot(sv, X_te_df, feature_names=X.columns.tolist(), plot_type='bar')

### Waterfall — Single Prediction Breakdown

In [ ]:
shap_exp  = explainer(X_te_df)
churn_idx = int(np.where(y_te == 1)[0][0])
print(f'Churned customer — predicted probability: {y_prob[churn_idx]:.4f}')
shap.plots.waterfall(shap_exp[churn_idx])

In [ ]:
no_churn_idx = int(np.where(y_te == 0)[0][0])
print(f'Retained customer — predicted probability: {y_prob[no_churn_idx]:.4f}')
shap.plots.waterfall(shap_exp[no_churn_idx])

### Dependence Plot — Top Feature

In [ ]:
top_idx  = int(np.abs(sv).mean(axis=0).argmax())
top_feat = X.columns[top_idx]
print(f'Top feature by mean |SHAP|: {top_feat}')
shap.dependence_plot(top_idx, sv, X_te_df, feature_names=X.columns.tolist())

## 7. Load Production Results

After running `python run_churn.py --n-trials 50`, inspect the saved predictions.

In [ ]:
preds = pd.read_csv(ROOT / 'data/processed/churn_predictions.csv')
print(f'Predictions: {len(preds):,} customers')
print(f'Predicted churn rate: {preds["predicted_churn"].mean():.1%}')
preds.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(preds.loc[preds['actual_churn']==0, 'churn_probability'],
        bins=40, alpha=0.6, color='steelblue', label='Retained', density=True)
ax.hist(preds.loc[preds['actual_churn']==1, 'churn_probability'],
        bins=40, alpha=0.6, color='salmon', label='Churned', density=True)
ax.set_xlabel('Predicted Churn Probability')
ax.set_ylabel('Density')
ax.set_title('Score Separation by Actual Churn Status')
ax.legend()
plt.tight_layout()
plt.show()